<a href="https://colab.research.google.com/github/SaharMian/recsys-letterboxd/blob/main/nfc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***monterar Drive, läser in den sparade splitten och definierar utvärderingen:***

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import torch

BASE = '/content/drive/MyDrive/recsys/'
SEED, LIKE, K = 42, 4.0, 10
device = 'cuda'
print("GPU:", torch.cuda.is_available())

data = pd.read_parquet(BASE + 'split_3000.parquet')
data['u'] = data['user_name'].cat.codes
data['i'] = data['film_id'].cat.codes
n_users, n_films = data['u'].nunique(), data['i'].nunique()

train = data[data['split'] == 'train']
val   = data[data['split'] == 'val']
test  = data[data['split'] == 'test']
seen_train = train.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def relevant_sets(df):
    liked = df[df['rating'] >= LIKE]
    return liked.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def evaluate(recommend_fn, eval_df, seen):
    relevant = relevant_sets(eval_df)
    ndcgs, recalls, recommended = [], [], set()
    for user, rel in relevant.items():
        recs = recommend_fn(user, seen.get(user, set()), K)
        recommended.update(recs)
        hits = [1 if f in rel else 0 for f in recs]
        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
        ndcgs.append(dcg / idcg)
        recalls.append(sum(hits) / len(rel))
    return {'NDCG@10': np.mean(ndcgs), 'Recall@10': np.mean(recalls),
            'Täckning': len(recommended) / data['film_id'].nunique()}

pop_rank = train[train['rating'] >= LIKE]['film_id'].value_counts().index.tolist()
def recommend_pop(user, seen, k):
    out = []
    for f in pop_rank:
        if f not in seen:
            out.append(f)
            if len(out) == k:
                break
    return out

print(evaluate(recommend_pop, val, seen_train))

Mounted at /content/drive
GPU: True
{'NDCG@10': np.float64(0.18405938695253699), 'Recall@10': np.float64(0.03855764705308532), 'Täckning': 0.001364633254812769}


***Gör om datan till gillar och skapa negativa exempel***

In [2]:
pos = train[train['rating'] >= LIKE]
pos_u = torch.tensor(pos['u'].values, dtype=torch.long, device=device)
pos_i = torch.tensor(pos['i'].values, dtype=torch.long, device=device)
print(f"Positiva exempel: {len(pos_u):,} ({len(pos_u) / len(train):.0%} av train)")

# Vilka (användare, film) har ett betyg i train, oavsett hur högt
seen_mask = torch.zeros(n_users, n_films, dtype=torch.bool, device=device)
seen_mask[torch.tensor(train['u'].values, dtype=torch.long, device=device),
          torch.tensor(train['i'].values, dtype=torch.long, device=device)] = True

def sample_negatives(u, n_neg=4):
    u = u.repeat_interleave(n_neg)
    neg = torch.randint(0, n_films, u.shape, device=device)
    bad = seen_mask[u, neg]
    while bad.any():
        neg[bad] = torch.randint(0, n_films, (int(bad.sum()),), device=device)
        bad = seen_mask[u, neg]
    return u, neg

# Test
torch.manual_seed(SEED)
u_neg, i_neg = sample_negatives(pos_u[:1_000_000])
print(f"Negativa exempel: {len(u_neg):,}")

film_count = torch.bincount(torch.tensor(train['i'].values, device=device), minlength=n_films)
print("Median antal betyg per film:  positiva",
      film_count[pos_i].float().median().item(),
      "| negativa", film_count[i_neg].float().median().item())

Positiva exempel: 1,333,735 (38% av train)
Negativa exempel: 4,000,000
Median antal betyg per film:  positiva 462.0 | negativa 11.0


***NeuMF-modellen och rekommendationsfunktionen,otränad***

In [3]:
import torch.nn as nn

class NeuMF(nn.Module):
    def __init__(self, n_users, n_films, d_gmf=32, d_mlp=32, layers=(64, 32, 16)):
        super().__init__()
        self.Pg, self.Qg = nn.Embedding(n_users, d_gmf), nn.Embedding(n_films, d_gmf)
        self.Pm, self.Qm = nn.Embedding(n_users, d_mlp), nn.Embedding(n_films, d_mlp)
        mlp, in_dim = [], 2 * d_mlp
        for h in layers:
            mlp += [nn.Linear(in_dim, h), nn.ReLU()]
            in_dim = h
        self.mlp = nn.Sequential(*mlp)
        self.out = nn.Linear(d_gmf + layers[-1], 1)
        for emb in (self.Pg, self.Qg, self.Pm, self.Qm):
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, u, i):
        g = self.Pg(u) * self.Qg(i)
        m = self.mlp(torch.cat([self.Pm(u), self.Qm(i)], dim=1))
        return self.out(torch.cat([g, m], dim=1)).squeeze(1)

def recommend_neumf(model, exclude=('train',), k=K, chunk=50):
    ex = data[data['split'].isin(exclude)]
    ex_u = torch.tensor(ex['u'].values, dtype=torch.long, device=device)
    ex_i = torch.tensor(ex['i'].values, dtype=torch.long, device=device)
    users = data['user_name'].cat.categories
    films = data['film_id'].cat.categories
    all_films = torch.arange(n_films, device=device)
    model.eval()
    recs = {}
    with torch.no_grad():
        for start in range(0, n_users, chunk):
            end = min(start + chunk, n_users)
            u = torch.arange(start, end, device=device)
            scores = model(u.repeat_interleave(n_films), all_films.repeat(len(u))).view(len(u), n_films)
            sel = (ex_u >= start) & (ex_u < end)
            scores[ex_u[sel] - start, ex_i[sel]] = float('-inf')
            top = scores.topk(k, dim=1).indices.cpu().numpy()
            for row, uid in enumerate(range(start, end)):
                recs[users[uid]] = [films[j] for j in top[row]]
    return recs

torch.manual_seed(SEED)
model = NeuMF(n_users, n_films).to(device)
recs = recommend_neumf(model)
print(evaluate(lambda user, seen, kk: recs[user], val, seen_train))

{'NDCG@10': np.float64(0.0007588090833367592), 'Recall@10': np.float64(0.00010598424943477677), 'Täckning': 0.009341239541873122}


***träna NFC**

In [4]:
import copy

N_NEG, BATCH, LR, EPOCHS = 4, 16384, 0.002, 10

torch.manual_seed(SEED)
model = NeuMF(n_users, n_films).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
best_ndcg, best_state, best_epoch = -1, None, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    u_neg, i_neg = sample_negatives(pos_u, N_NEG)
    u_all = torch.cat([pos_u, u_neg])
    i_all = torch.cat([pos_i, i_neg])
    y_all = torch.cat([torch.ones(len(pos_u)), torch.zeros(len(u_neg))]).to(device)
    perm = torch.randperm(len(y_all), device=device)

    total = 0.0
    for s in range(0, len(perm), BATCH):
        idx = perm[s:s + BATCH]
        loss = loss_fn(model(u_all[idx], i_all[idx]), y_all[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)

    recs = recommend_neumf(model)
    res = evaluate(lambda user, seen, kk: recs[user], val, seen_train)
    print(f"Epok {epoch:>2}: loss {total / len(perm):.4f} | val NDCG {res['NDCG@10']:.4f}, "
          f"Recall {res['Recall@10']:.4f}, Täckning {res['Täckning']:.4f}")
    if res['NDCG@10'] > best_ndcg:
        best_ndcg, best_state, best_epoch = res['NDCG@10'], copy.deepcopy(model.state_dict()), epoch

model.load_state_dict(best_state)
torch.save(best_state, BASE + 'ncf_model.pt')
print(f"Bästa epok: {best_epoch} (val NDCG {best_ndcg:.4f}), sparad till Drive")

Epok  1: loss 0.2911 | val NDCG 0.1941, Recall 0.0411, Täckning 0.0039
Epok  2: loss 0.1985 | val NDCG 0.2015, Recall 0.0443, Täckning 0.0103
Epok  3: loss 0.1770 | val NDCG 0.2185, Recall 0.0484, Täckning 0.0137
Epok  4: loss 0.1585 | val NDCG 0.2261, Recall 0.0505, Täckning 0.0253
Epok  5: loss 0.1409 | val NDCG 0.2303, Recall 0.0528, Täckning 0.0354
Epok  6: loss 0.1273 | val NDCG 0.2297, Recall 0.0527, Täckning 0.0433
Epok  7: loss 0.1188 | val NDCG 0.2174, Recall 0.0505, Täckning 0.0555
Epok  8: loss 0.1127 | val NDCG 0.2219, Recall 0.0510, Täckning 0.0592
Epok  9: loss 0.1083 | val NDCG 0.2206, Recall 0.0507, Täckning 0.0625
Epok 10: loss 0.1050 | val NDCG 0.2143, Recall 0.0492, Täckning 0.0705
Bästa epok: 5 (val NDCG 0.2303), sparad till Drive


In [5]:
recs_ncf = recommend_neumf(model)
r = evaluate(lambda user, seen, kk: recs_ncf[user], val, seen_train)

tbl = pd.read_csv(BASE + 'results_val.csv', index_col=0)
tbl.loc[f'NCF (NeuMF, epok {best_epoch})'] = [round(r['NDCG@10'], 4),
                                               round(r['Recall@10'], 4),
                                               round(r['Täckning'], 4)]
tbl.to_csv(BASE + 'results_val.csv')
print(tbl)

tc = train['film_id'].value_counts().to_dict()
print("\nMedian antal betyg per rekommenderad film (NCF):",
      np.median([tc[f] for rec in recs_ncf.values() for f in rec]))

                       NDCG@10  Recall@10  Täckning
Popularitet (gillade)   0.1841     0.0386    0.0014
SVD (min 0 betyg)       0.0086     0.0022    0.0323
SVD (min 1000 betyg)    0.1757     0.0365    0.0038
SVD (min 1500 betyg)    0.1929     0.0406    0.0010
NCF (NeuMF, epok 5)     0.2303     0.0528    0.0354

Median antal betyg per rekommenderad film (NCF): 1202.0
